In [12]:
# Semente fixa para reprodutibilidade
RANDOM_STATE = 42


# 02 — Pré-processamento e Engenharia de Atributos

Grupo 50 • Tech Challenge — Fase 2

## Objetivo
Integrar as bases, construir o `TARGET`, tratar tipos e valores ausentes, criar atributos derivados e preparar a tabela de modelagem. A identificação de perfis repetidos é preservada para o agrupamento da divisão entre treino, validação e teste.

> **Importante:** a imputação numérica e a padronização não são realizadas neste notebook. Elas devem ser aprendidas exclusivamente a partir do conjunto de treino dentro do pipeline de modelagem, evitando vazamento de informação.


In [13]:
from pathlib import Path
import pandas as pd
import numpy as np

# Localiza as bases tanto no repositório local quanto no Colab.
candidatos = [
    Path.cwd().parent / "data" / "raw",
    Path.cwd() / "data" / "raw",
    Path("/content/data/raw"),
]

RAW = next((p for p in candidatos if (p / "application_record.csv").exists()), None)
if RAW is None and Path("/content/application_record.csv").exists():
    RAW = Path("/content")
if RAW is None:
    raise FileNotFoundError("Não foi possível localizar application_record.csv e credit_record.csv.")

ROOT = RAW.parent.parent if RAW.name == "raw" else RAW
PROC = ROOT / "data" / "processed"
PROC.mkdir(parents=True, exist_ok=True)

app = pd.read_csv(RAW / "application_record.csv")
credit = pd.read_csv(RAW / "credit_record.csv")

print("Diretório das bases:", RAW)
print("Application:", app.shape)
print("Credit:", credit.shape)


Diretório das bases: c:\projetos\Grupo_50_Tech-Challenge-Fase-2\data\raw
Application: (438557, 18)
Credit: (1048575, 3)


## 1. Construção do TARGET

O `TARGET` é definido a partir do pior status numérico observado no histórico de crédito. Status `2`, `3`, `4` e `5` representam `TARGET = 1` (mau pagador); os demais casos ficam em `TARGET = 0`.


In [14]:
# Evita que duplicidades cadastrais da base de aplicação criem múltiplas linhas.
application_clean = app.drop_duplicates("ID", keep="first").copy()

credit["STATUS_NUM"] = pd.to_numeric(credit["STATUS"], errors="coerce")
worst_status = credit.groupby("ID")["STATUS_NUM"].max().rename("WORST_STATUS")
target = worst_status.isin([2, 3, 4, 5]).astype(int).rename("TARGET")

df = application_clean.merge(
    target, left_on="ID", right_index=True, how="inner"
)

print("Aplicações originais:", len(app))
print("Aplicações únicas por ID:", len(application_clean))
print("Base final:", df.shape)
display(
    df["TARGET"].value_counts()
      .rename(index={0: "Bom pagador", 1: "Mau pagador"})
      .to_frame("quantidade")
)
display(
    (df["TARGET"].value_counts(normalize=True) * 100)
      .rename(index={0: "Bom pagador", 1: "Mau pagador"})
      .round(2).to_frame("percentual")
)


Aplicações originais: 438557
Aplicações únicas por ID: 438510
Base final: (36457, 19)


,quantidade
TARGET,
Bom pagador,35841
Mau pagador,616


,percentual
TARGET,
Bom pagador,98.31
Mau pagador,1.69


## 2. Engenharia de atributos

- `AGE_YEARS`: idade em anos a partir de `DAYS_BIRTH`.
- `EMPLOYED_YEARS`: tempo de emprego em anos.
- `DAYS_EMPLOYED = 365243` é tratado como valor especial/ausente.
- Identificadores e as variáveis brutas usadas para derivar os atributos são removidos da matriz de modelagem.


In [15]:
df_model = df.copy()

df_model["AGE_YEARS"] = (
    -df_model["DAYS_BIRTH"] / 365.25
).round(2)

df_model["EMPLOYED_YEARS"] = (
    df_model["DAYS_EMPLOYED"]
    .where(df_model["DAYS_EMPLOYED"] < 365243, np.nan)
    .abs()
    .div(365.25)
    .round(2)
)

drop_cols = ["ID", "DAYS_BIRTH", "DAYS_EMPLOYED"]
df_model = df_model.drop(
    columns=[c for c in drop_cols if c in df_model.columns]
)

# Mantém as categóricas explicitamente tratadas,
# mas não imputa numéricas aqui.
# A mediana deve ser aprendida somente no treino
# dentro do pipeline de modelagem.
cat_cols = df_model.select_dtypes(
    include=["object"]
).columns.tolist()

num_cols = [
    c for c in df_model.columns
    if c not in cat_cols + ["TARGET"]
]

for c in cat_cols:
    df_model[c] = df_model[c].fillna("UNKNOWN")

for c in num_cols:
    df_model[c] = pd.to_numeric(
        df_model[c],
        errors="coerce"
    )

print("Variáveis categóricas:", cat_cols)
print("Quantidade de variáveis numéricas:", len(num_cols))
print(
    "Nulos numéricos mantidos para imputação no pipeline:",
    int(df_model[num_cols].isna().sum().sum())
)

display(df_model.head())


Variáveis categóricas: ['CODE_GENDER', 'FLAG_OWN_CAR', 'FLAG_OWN_REALTY', 'NAME_INCOME_TYPE', 'NAME_EDUCATION_TYPE', 'NAME_FAMILY_STATUS', 'NAME_HOUSING_TYPE', 'OCCUPATION_TYPE']
Quantidade de variáveis numéricas: 9
Nulos numéricos mantidos para imputação no pipeline: 6135


,CODE_GENDER,FLAG_OWN_CAR,FLAG_OWN_REALTY,CNT_CHILDREN,AMT_INCOME_TOTAL,NAME_INCOME_TYPE,NAME_EDUCATION_TYPE,NAME_FAMILY_STATUS,NAME_HOUSING_TYPE,FLAG_MOBIL,FLAG_WORK_PHONE,FLAG_PHONE,FLAG_EMAIL,OCCUPATION_TYPE,CNT_FAM_MEMBERS,TARGET,AGE_YEARS,EMPLOYED_YEARS
0,M,Y,Y,0,427500.0,Working,Higher education,Civil marriage,Rented apartment,1,1,0,0,UNKNOWN,2.0,0,32.87,12.44
1,M,Y,Y,0,427500.0,Working,Higher education,Civil marriage,Rented apartment,1,1,0,0,UNKNOWN,2.0,0,32.87,12.44
2,M,Y,Y,0,112500.0,Working,Secondary / secondary special,Married,House / apartment,1,0,0,0,Security staff,2.0,0,58.79,3.10
3,F,N,Y,0,270000.0,Commercial associate,Secondary / secondary special,Single / not married,House / apartment,1,0,1,1,Sales staff,1.0,0,52.32,8.35
4,F,N,Y,0,270000.0,Commercial associate,Secondary / secondary special,Single / not married,House / apartment,1,0,1,1,Sales staff,1.0,0,52.32,8.35


## 3. Remoção de variáveis constantes

Variáveis sem variação não carregam informação para classificação e são removidas antes da modelagem.

In [16]:
feature_cols = [
    c for c in df_model.columns
    if c != "TARGET"
]

constant_cols = [
    c for c in feature_cols
    if df_model[c].nunique(dropna=False) <= 1
]

df_model = df_model.drop(
    columns=constant_cols
)

print(
    "Variáveis constantes removidas:",
    constant_cols if constant_cols else "nenhuma"
)

print(
    "Dimensão final da tabela de modelagem:",
    df_model.shape
)

Variáveis constantes removidas: ['FLAG_MOBIL']
Dimensão final da tabela de modelagem: (36457, 17)


## 4. Identificação dos perfis cadastrais

O agrupamento é definido pelas características cadastrais, sem `ID` e sem `TARGET`. Assim, registros com o mesmo perfil permanecem no mesmo grupo durante a divisão posterior. O agrupamento é criado antes de qualquer imputação numérica para não incorporar estatísticas calculadas a partir de outras observações.

In [17]:
group_cols = [c for c in df_model.columns if c != "TARGET"]

# Representação determinística do perfil; NaN permanece distinguível de valor preenchido.
profile_key = df_model[group_cols].astype(object).where(df_model[group_cols].notna(), "<NA>").astype(str).agg("||".join, axis=1)
df_model["PROFILE_GROUP"] = pd.factorize(profile_key, sort=True)[0]

conflicts = df_model.groupby("PROFILE_GROUP")["TARGET"].nunique()
print("Perfis/grupos distintos:", df_model["PROFILE_GROUP"].nunique())
print("Grupos com TARGET conflitante:", int((conflicts > 1).sum()))
print("Maior tamanho de grupo:", int(df_model["PROFILE_GROUP"].value_counts().max()))


Perfis/grupos distintos: 9723
Grupos com TARGET conflitante: 269
Maior tamanho de grupo: 51


## 5. Verificações finais

A tabela abaixo é a saída que será consumida pelo notebook de modelagem. A coluna `PROFILE_GROUP` será usada apenas para impedir sobreposição de perfis entre treino, validação e teste; ela **não é uma feature do modelo**.

In [18]:
print("TARGET nulos:", int(df_model["TARGET"].isna().sum()))
print("PROFILE_GROUP nulos:", int(df_model["PROFILE_GROUP"].isna().sum()))
print("Features disponíveis:", df_model.drop(columns=["TARGET", "PROFILE_GROUP"]).shape[1])

display(
    pd.DataFrame({
        "tipo": df_model.drop(columns=["TARGET", "PROFILE_GROUP"]).dtypes.astype(str),
        "nulos": df_model.drop(columns=["TARGET", "PROFILE_GROUP"]).isna().sum(),
        "n_unicos": df_model.drop(columns=["TARGET", "PROFILE_GROUP"]).nunique(dropna=False),
    }).sort_values("nulos", ascending=False).head(15)
)


TARGET nulos: 0
PROFILE_GROUP nulos: 0
Features disponíveis: 16


,tipo,nulos,n_unicos
EMPLOYED_YEARS,float64,6135,1887
CODE_GENDER,object,0,2
FLAG_OWN_REALTY,object,0,2
FLAG_OWN_CAR,object,0,2
AMT_INCOME_TOTAL,float64,0,265
NAME_INCOME_TYPE,object,0,5
NAME_EDUCATION_TYPE,object,0,5
CNT_CHILDREN,int64,0,9
NAME_FAMILY_STATUS,object,0,5
NAME_HOUSING_TYPE,object,0,6


In [19]:
# Artefato local para os notebooks seguintes.
# data/processed/ está no .gitignore e não será versionado.
output_path = PROC / "base_modelagem.csv"
df_model.to_csv(output_path, index=False)
print("Base de modelagem salva em:", output_path)
print("Shape salvo:", df_model.shape)


Base de modelagem salva em: c:\projetos\Grupo_50_Tech-Challenge-Fase-2\data\processed\base_modelagem.csv
Shape salvo: (36457, 18)


## Conclusão

O notebook entrega uma base de modelagem preparada para a etapa seguinte, mantendo o `TARGET` separado das features e o `PROFILE_GROUP` exclusivamente para a divisão agrupada. A imputação numérica e a escala ficam para o pipeline de modelagem, onde os parâmetros podem ser aprendidos apenas com os dados de treino.
